# Matched-Budget No-Exp Control

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR      = '/content/drive/MyDrive/GNN_MEA/'
VICTIM_DIR    = os.path.join(BASE_DIR, 'victim_models')
EXPLAINER_DIR = os.path.join(BASE_DIR, 'explainers')
RESULTS_DIR   = os.path.join(BASE_DIR, 'results')
CACHE_DIR     = os.path.join(BASE_DIR, 'matched_noexp_cache')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

RESULTS_FILE = os.path.join(RESULTS_DIR, 'matched_noexp_results.json')

Mounted at /content/drive


In [ ]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.0 MB/s eta 0:00:00


In [ ]:
import copy, json, random, time
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_mean_pool
from torch_geometric.explain import Explainer, PGExplainer, GNNExplainer
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Experiment configuration

In [ ]:
SEEDS = [42]
ARCH = 'GCN'
EXPLAINER_NAME = 'GNN'    # GNNExplainer

MAX_ITERS  = 10
TOP_K      = 3
N_MC       = 5
DELTA_MAX  = 20
P_FLIP     = 0.05
N_RANDOM   = 10
N_CARRY    = 5

DATASETS = {
    'PTC_FM':             dict(target_n_train=60,  max_samples=140, enabled=True),
    'NCI1':               dict(target_n_train=560, max_samples=574, enabled=True),
    'Tox21_AhR_training': dict(target_n_train=100, max_samples=300, enabled=True),
}
DATASETS = {k: v for k, v in DATASETS.items() if v['enabled']}

BATCH_PREDICT = 512

## 2. Datasets, splits, victims, explainer

In [ ]:
datasets = {}
for name in DATASETS:
    ds = TUDataset(root=f'data/{name}', name=name)
    datasets[name] = ds
    assert ds[0].x is not None, f"{name}: features are None"
    print(f"{name}: {len(ds)} graphs, {ds.num_classes} classes, "
          f"feature dim = {ds[0].x.shape[1]}")

Processing...
Done!


PTC_FM: 349 graphs, 2 classes, feature dim = 18


Processing...
Done!


NCI1: 4110 graphs, 2 classes, feature dim = 37


Processing...


Tox21_AhR_training: 8169 graphs, 2 classes, feature dim = 50


Done!


In [ ]:
def split_dataset(dataset, seed=42):
    labels = [data.y.item() for data in dataset]
    train_idx, remaining_idx = train_test_split(
        range(len(dataset)), test_size=0.4, stratify=labels, random_state=seed)
    remaining_labels = [labels[i] for i in remaining_idx]
    shadow_idx, test_idx = train_test_split(
        remaining_idx, test_size=0.5, stratify=remaining_labels, random_state=seed)
    return train_idx, shadow_idx, test_idx

def get_feature_dim(dataset):
    return dataset[0].x.shape[1]

dataset_splits = {}
for name in DATASETS:
    tr, sh, te = split_dataset(datasets[name])
    dataset_splits[name] = {'train': tr, 'shadow': sh, 'test': te}
    print(f"{name}: {len(tr)} train / {len(sh)} shadow / {len(te)} test")

PTC_FM: 209 train / 70 shadow / 70 test
NCI1: 2466 train / 822 shadow / 822 test
Tox21_AhR_training: 4901 train / 1634 shadow / 1634 test


In [ ]:
class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)

class GAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim // 8, heads=8)
        self.conv2 = GATConv(hidden_dim, hidden_dim // 8, heads=8)
        self.conv3 = GATConv(hidden_dim, hidden_dim // 8, heads=8)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)

class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.conv3 = SAGEConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)

MODEL_CLASSES = {'GCN': GCN, 'GAT': GAT, 'GraphSAGE': GraphSAGE}

In [ ]:
def load_victim(name, arch, device='cuda'):
    save_path = os.path.join(VICTIM_DIR, arch, f'{name}_victim.pt')
    ckpt = torch.load(save_path, map_location=device, weights_only=False)
    model = MODEL_CLASSES[arch](ckpt['in_dim'], ckpt['config']['hidden_dim'],
                                ckpt['num_classes']).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, ckpt

victims = {}
for name in DATASETS:
    m, ck = load_victim(name, ARCH, device)
    victims[name] = m
    print(f"{name}/{ARCH}: acc={ck['accuracy']:.4f}, config={ck['config']}")

PTC_FM/GCN: acc=0.6571, config={'hidden_dim': 128, 'epochs': 500}
NCI1/GCN: acc=0.6727, config={'hidden_dim': 64, 'epochs': 1000}
Tox21_AhR_training/GCN: acc=0.8898, config={'hidden_dim': 64, 'epochs': 700}


In [ ]:
def create_gnn_explainer(model):
    model_copy = copy.deepcopy(model)
    for p in model_copy.parameters():
        p.requires_grad_(True)
    return Explainer(
        model=model_copy,
        algorithm=GNNExplainer(epochs=100, lr=0.01),
        explanation_type='phenomenon',
        node_mask_type='object',
        edge_mask_type=None,
        model_config=dict(mode='multiclass_classification',
                          task_level='graph', return_type='raw'),
    )

def get_explanation(explainer, model, data, device='cuda'):
    data = data.to(device)
    batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
    target = model(data.x, data.edge_index, batch).argmax(dim=1)
    explanation = explainer(data.x, data.edge_index, target=target, batch=batch)

    if getattr(explanation, 'node_mask', None) is not None:
        nm = explanation.node_mask
        imp = nm.mean(dim=1) if nm.dim() > 1 else nm
        important_nodes = (imp > imp.median()).nonzero(as_tuple=True)[0]
        return important_nodes.cpu()
    elif getattr(explanation, 'edge_mask', None) is not None:
        em = explanation.edge_mask
        important_edges = (em > em.median()).nonzero(as_tuple=True)[0]
        return torch.unique(data.edge_index[:, important_edges].flatten()).cpu()
    raise ValueError("No mask found in explanation")

explainers = {name: create_gnn_explainer(victims[name]) for name in DATASETS}
print("GNNExplainer ready for:", list(explainers))

GNNExplainer ready for: ['PTC_FM', 'NCI1', 'Tox21_AhR_training']


## 3. Query ledger

In [ ]:
class BudgetExhausted(Exception):
    pass

class QueryLedger:
    def __init__(self, max_queries=None, start=0):
        self.api_calls = start
        self.explanations_consumed = 0
        self.max_queries = max_queries
        self.ceiling_hit = False

    def exhausted(self):
        if self.max_queries is not None and self.api_calls >= self.max_queries:
            self.ceiling_hit = True
            return True
        return False

    def _forward(self, model, data_list, device):
        model.eval()
        preds = []
        with torch.no_grad():
            for batch in DataLoader(data_list, batch_size=BATCH_PREDICT):
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index, batch.batch)
                preds.extend(out.argmax(dim=1).cpu().tolist())
        return preds

    def predict(self, model, data, device):
        self.api_calls += 1
        return self._forward(model, [data], device)[0]

    def predict_many(self, model, data_list, device):
        if not data_list:
            return []
        self.api_calls += len(data_list)
        return self._forward(model, data_list, device)

    def explain(self, explainer, model, data, device):
        self.explanations_consumed += 1
        return get_explanation(explainer, model, data, device)

## 4. Graph helpers

In [ ]:
def get_edge_set(edge_index):
    e = edge_index.cpu().numpy()
    return {(min(int(a), int(b)), max(int(a), int(b))) for a, b in zip(e[0], e[1])}

def edges_to_index(edge_set):
    if not edge_set:
        return torch.zeros((2, 0), dtype=torch.long)
    rows, cols = [], []
    for (u, v) in edge_set:
        rows += [u, v]; cols += [v, u]
    return torch.tensor([rows, cols], dtype=torch.long)

def graph_from_edges(template, edge_set):
    g = template.clone()
    g.edge_index = edges_to_index(edge_set)
    return g

def toggle(edge_set, u, v):
    key = (min(u, v), max(u, v))
    out = set(edge_set)
    out.discard(key) if key in out else out.add(key)
    return out

def perturb_edge_set(edge_set, n_nodes, p_flip, exclude, rng):
    iu = np.triu_indices(n_nodes, k=1)
    draws = rng.random(len(iu[0])) < p_flip
    out = set(edge_set)
    ex = (min(exclude), max(exclude)) if exclude is not None else None
    for a, b in zip(iu[0][draws], iu[1][draws]):
        key = (int(a), int(b))
        if key == ex:
            continue
        out.discard(key) if key in out else out.add(key)
    return out

## 5. Phase 1

In [ ]:
def run_phase1(dataset, shadow_idx, model, ledger, device):
    pairs, non_boundary = [], []
    t0 = time.time()
    for n, i in enumerate(shadow_idx):
        data = dataset[i]
        orig_pred = ledger.predict(model, data, device)
        base_edges = sorted(get_edge_set(data.edge_index))

        variants, kept = [], []
        for (u, v) in base_edges:
            cand = set(base_edges); cand.discard((u, v))
            if not cand:
                continue
            variants.append(graph_from_edges(data, cand)); kept.append((u, v))

        if not variants:
            non_boundary.append(i); continue

        preds = ledger._forward(model, variants, device)   # compute, do not charge
        hit = next((j for j, p in enumerate(preds) if p != orig_pred), None)
        ledger.api_calls += (hit + 1) if hit is not None else len(variants)

        if hit is not None:
            o = data.clone(); o.y = torch.tensor([orig_pred])
            g = variants[hit];  g.y = torch.tensor([preds[hit]])
            pairs.append((o.cpu(), g.cpu()))
        else:
            non_boundary.append(i)

        if (n + 1) % 200 == 0:
            print(f"    [Phase 1 | shared] {n+1}/{len(shadow_idx)} graphs, "
                  f"{len(pairs)} pairs, {ledger.api_calls:,} queries, "
                  f"{time.time()-t0:.0f}s")

    return pairs, non_boundary

## 6. Candidate construction

In [ ]:
def build_candidates_guided(edge_set, n_nodes, important_nodes,
                            carry_over, n_random, rng):
    """Eq. (9): within-explanation, explanation-to-neighbour, carry-over, random."""
    imp = set(int(x) for x in important_nodes.tolist())
    neighbours = set()
    for (u, v) in edge_set:
        if u in imp and v not in imp: neighbours.add(v)
        if v in imp and u not in imp: neighbours.add(u)

    cands = set()
    imp_sorted = sorted(imp)
    for a in range(len(imp_sorted)):
        for b in range(a + 1, len(imp_sorted)):
            cands.add((imp_sorted[a], imp_sorted[b]))
    for u in imp:
        for v in neighbours:
            if u != v:
                cands.add((min(u, v), max(u, v)))
    if carry_over:
        cands.update(carry_over)
    for _ in range(n_random):
        u, v = rng.integers(0, n_nodes, size=2)
        if u != v:
            cands.add((min(int(u), int(v)), max(int(u), int(v))))
    return cands


def build_candidates_random(n_nodes, target_size, carry_over, rng):
    cands = set(carry_over) if carry_over else set()
    max_pairs = n_nodes * (n_nodes - 1) // 2
    budget = min(target_size, max_pairs)
    guard = 0
    while len(cands) < budget and guard < 50 * budget + 100:
        u, v = rng.integers(0, n_nodes, size=2)
        guard += 1
        if u != v:
            cands.add((min(int(u), int(v)), max(int(u), int(v))))
    return cands

## 7. Sensitivity estimation

In [ ]:
def score_candidates(data, edge_set, candidates, model, orig_pred,
                     ledger, n_mc, p_flip, rng, device):
    cand_list = sorted(candidates)
    graphs, owner = [], []
    n = data.num_nodes

    for ci, (u, v) in enumerate(cand_list):
        for _ in range(n_mc):
            base = perturb_edge_set(edge_set, n, p_flip, (u, v), rng)
            flip = toggle(base, u, v)
            if not base or not flip:
                continue
            graphs.append(graph_from_edges(data, base));  owner.append((ci, 'base'))
            graphs.append(graph_from_edges(data, flip));  owner.append((ci, 'flip'))

    preds = ledger.predict_many(model, graphs, device)

    acc = defaultdict(lambda: [0.0, 0])
    pending = {}
    for (ci, kind), p in zip(owner, preds):
        if kind == 'base':
            pending[ci] = int(p != orig_pred)
        else:
            if ci in pending:
                acc[ci][0] += int(p != orig_pred) - pending.pop(ci)
                acc[ci][1] += 1

    return {cand_list[ci]: (s / c if c else 0.0) for ci, (s, c) in acc.items()}

## 8. Phase 2

In [ ]:
def boundary_search(data, model, ledger, rng, device,
                    explainer=None, n_candidates=None,
                    max_iters=MAX_ITERS, top_k=TOP_K, n_mc=N_MC,
                    delta_max=DELTA_MAX, p_flip=P_FLIP,
                    n_random=N_RANDOM, n_carry=N_CARRY):
    assert (explainer is None) != (n_candidates is None)

    orig_pred = ledger.predict(model, data, device)
    orig_edges = get_edge_set(data.edge_index)
    cur_edges = set(orig_edges)
    carry_over, cand_sizes, full_space, vsub_frac = set(), [], [], []

    for t in range(max_iters):
        if ledger.exhausted():
            return None, {'iters': t, 'cand_sizes': cand_sizes,
                            'full_space': full_space, 'vsub_frac': vsub_frac,
                            'exit': 'ceiling'}

        cur = graph_from_edges(data, cur_edges)
        cur_pred = ledger.predict(model, cur, device)
        delta_e = len(orig_edges.symmetric_difference(cur_edges))

        if cur_pred != orig_pred and delta_e <= delta_max:
            cur.y = torch.tensor([cur_pred])
            return cur.cpu(), {'iters': t + 1, 'cand_sizes': cand_sizes,
                               'full_space': full_space, 'vsub_frac': vsub_frac,
                               'exit': 'success', 'delta_e': delta_e}
        if delta_e >= delta_max:
            return None, {'iters': t + 1, 'cand_sizes': cand_sizes, 'full_space': full_space,
            'vsub_frac': vsub_frac, 'exit': 'budget'}

        if explainer is not None:
            important = ledger.explain(explainer, model, cur, device)
            vsub_frac.append(len(important) / max(data.num_nodes, 1))
            candidates = build_candidates_guided(cur_edges, data.num_nodes,
                                                 important, carry_over,
                                                 n_random, rng)
        else:
            size = (n_candidates[min(t, len(n_candidates) - 1)]
                    if isinstance(n_candidates, (list, tuple)) else n_candidates)
            candidates = build_candidates_random(data.num_nodes, int(size),
                                                 carry_over, rng)
        cand_sizes.append(len(candidates))
        full_space.append(data.num_nodes * (data.num_nodes - 1) // 2)

        sens = score_candidates(data, cur_edges, candidates, model, orig_pred,
                                ledger, n_mc, p_flip, rng, device)

        ranked = sorted(sens.items(), key=lambda kv: -kv[1])
        top = [e for e, s in ranked[:top_k] if s > 0]
        if not top:
            return None, {'iters': t + 1, 'cand_sizes': cand_sizes,
                          'full_space': full_space, 'vsub_frac': vsub_frac,
                          'exit': 'stagnation'}

        for (u, v) in top:
            cur_edges = toggle(cur_edges, u, v)
        if not cur_edges:
            return None, {'iters': t + 1, 'cand_sizes': cand_sizes,
                          'full_space': full_space, 'vsub_frac': vsub_frac,
                          'exit': 'empty'}
        carry_over = {e for e, s in ranked[:n_carry] if s > 0}

    return None, {'iters': max_iters, 'cand_sizes': cand_sizes, 'full_space': full_space,
            'vsub_frac': vsub_frac, 'exit': 'cap'}

## 9. Arm runner

In [ ]:
def run_arm(arm, name, model, explainer, phase1_pairs, phase1_queries,
            non_boundary, dataset, cfg, n_candidates, seed, device):
    label = 'Boundary' if arm == 'guided' else 'No-Exp'
    rng = np.random.default_rng(seed)
    ledger = QueryLedger(max_queries=None, start=phase1_queries)

    produced = [(o, f, phase1_queries) for (o, f) in phase1_pairs]
    stats = {'exits': defaultdict(int), 'iters_success': [], 'cand_sizes': [],
             'full_space': [], 'vsub_frac': [], 'cand_by_graph': {},
             'stopped_reason': 'shadow_exhausted', 'wall_s': 0.0}

    t0 = time.time()
    for n, i in enumerate(non_boundary):
        if len(produced) * 2 >= cfg['max_samples']:
            stats['stopped_reason'] = 'max_samples'; break
        if ledger.exhausted():
            stats['stopped_reason'] = 'query_ceiling'; break
        data = dataset[i]
        # n_candidates is a {graph index -> [per-iteration sizes]} map for the
        # control, so it scores the same number of candidates the guided method
        # scored on this graph. Missing graphs fall back via the defaultdict.
        plan = n_candidates[i] if isinstance(n_candidates, dict) else n_candidates
        res, info = boundary_search(
            data, model, ledger, rng, device,
            explainer=explainer if arm == 'guided' else None,
            n_candidates=None if arm == 'guided' else plan)
        stats['exits'][info['exit']] += 1
        stats['cand_sizes'].extend(info['cand_sizes'])
        stats['cand_by_graph'][i] = list(info['cand_sizes'])
        stats['full_space'].extend(info.get('full_space', []))
        stats['vsub_frac'].extend(info.get('vsub_frac', []))
        if res is not None:
            o = data.clone()
            o.y = torch.tensor([ledger.predict(model, data, device)])
            produced.append((o.cpu(), res, ledger.api_calls))
            stats['iters_success'].append(info['iters'])

        if (n + 1) % 100 == 0:
            print(f"    [Phase 2 | {label}] {n+1}/{len(non_boundary)} graphs, "
                  f"pairs={len(produced)} n_train={len(produced)*2} "
                  f"queries={ledger.api_calls:,} {time.time()-t0:.0f}s")

    stats['wall_s'] = round(time.time() - t0, 1)
    return produced, ledger, stats


def truncate_at_n(produced, target_n):
    """Pairs in production order until n_train reaches target_n.
    Returns (samples, queries_needed, reached). The queries value also sets
    Q_cap; the samples are the matched-n_train training set."""
    out, q = [], 0
    for (o, f, cq) in produced:
        if len(out) + 2 > target_n:
            break
        out += [o, f]; q = cq
    return out, q, len(out) >= target_n


def truncate_at_q(produced, q_cap):
    out = []
    for (o, f, cq) in produced:
        if cq > q_cap:
            break
        out += [o, f]
    return out

## 10. Surrogate training and evaluation

In [ ]:
def train_surrogate(train_data, in_dim, num_classes, arch=ARCH, device='cuda',
                    seed=0):
    cpu_state = torch.get_rng_state()
    cuda_state = (torch.cuda.get_rng_state_all()
                  if torch.cuda.is_available() else None)
    try:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        g = torch.Generator(); g.manual_seed(seed)

        model = MODEL_CLASSES[arch](in_dim, 64, num_classes, dropout=0.0).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.CrossEntropyLoss()

        clean = []
        for d in train_data:
            d = d.cpu(); d.y = d.y.long(); clean.append(d)
        loader = DataLoader(clean, batch_size=8, shuffle=True, generator=g)

        model.train()
        for _ in range(500):
            for batch in loader:
                batch = batch.to(device)
                loss = loss_fn(model(batch.x, batch.edge_index, batch.batch), batch.y)
                loss.backward(); optimizer.step(); optimizer.zero_grad()
        return model
    finally:
        torch.set_rng_state(cpu_state)
        if cuda_state is not None:
            torch.cuda.set_rng_state_all(cuda_state)


def get_balanced_test_idx(victim, dataset, test_idx, device='cuda', seed=42):
    by_class = {}
    victim.eval()
    with torch.no_grad():
        for idx in test_idx:
            data = dataset[idx].to(device)
            batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
            pred = victim(data.x, data.edge_index, batch).argmax(1).item()
            by_class.setdefault(pred, []).append(idx)
    min_count = min(len(v) for v in by_class.values())
    rng = np.random.RandomState(seed)
    out = []
    for cls in sorted(by_class):
        out.extend(rng.choice(by_class[cls], min_count, replace=False))
    return out


def evaluate(surrogate, victim, dataset, balanced_idx, device='cuda'):
    surrogate.eval(); victim.eval()
    fid = acc = total = 0
    with torch.no_grad():
        for idx in balanced_idx:
            data = dataset[idx].to(device)
            batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
            s = surrogate(data.x, data.edge_index, batch).argmax(1).item()
            v = victim(data.x, data.edge_index, batch).argmax(1).item()
            fid += (s == v); acc += (s == data.y.item()); total += 1
    return round(fid / total, 4), round(acc / total, 4)

## 10b. Per-arm summary printer

In [ ]:
def print_arm_summary(row):
    d, arm = row['dataset'], row['arm']
    print(f"\n  {'-'*64}")
    print(f"  RESULT  {d} / {row['arch']} / {row['explainer']} / {arm} / seed {row['seed']}")
    print(f"  {'-'*64}")

    src = row.get('search_stats_source', arm)
    if src != arm:
        print(f"    NOTE   {arm} is derived from the {src} log, so the search "
              f"diagnostics below")
        print(f"           (queries, explanations, exits, iters/success) are "
              f"{src}'s and do not describe {arm}.")

    line = (f"    pairs            phase1={row['n_pairs_phase1']}  "
            f"phase2={row['n_pairs_phase2']}")
    if row.get('n_random_shadow_pairs'):
        line += f"  random_shadow={row['n_random_shadow_pairs']}"
    line += (f"  total={row['n_pairs_total']}"
             f"  (n_train max {row['n_pairs_total']*2})")
    print(line)

    print(f"    queries          phase1={row['queries_phase1']:,}  "
          f"total={row['queries_total']:,}")
    if row['explanations_consumed']:
        rate = row.get('explanation_rate') or 0.0
        per = int(1 / rate) if rate > 0 else None
        print(f"    explanations     {row['explanations_consumed']:,}"
              + (f"  (1 per {per:,} calls)" if per else ""))
    else:
        print(f"    explanations     0")
    print(f"    phase 2          attempted={row['graphs_attempted_phase2']}  "
          f"success_rate={row['phase2_success_rate']}")
    print(f"    exits            {row['exit_counts']}")
    mi = row.get('mean_iters_to_success')
    print(f"    iters/success    {mi:.3f}" if mi is not None
          else "    iters/success    n/a")
    if row.get('mean_vsub_frac'):
        print(f"    mean |Vsub|/|V|  {row['mean_vsub_frac']:.3f}")
    print(f"    stopped          {row['stopped_reason']}  "
          f"wall={row['wall_seconds']}s")
    print(f"    reached target   {row['reached_target']}  "
          f"queries_to_target={row['queries_to_target']:,}")

    mn = row.get('matched_n_train')
    if mn:
        print(f"    @ matched n_train={mn['n_train']:<5d} "
              f"fidelity={mn['fidelity']:.4f}  accuracy={mn['accuracy']:.4f}")
    mq = row.get('matched_queries')
    if mq:
        print(f"    @ matched Q={row['q_cap']:<10,} n_train={mq['n_train']:<5d} "
              f"fidelity={mq['fidelity']:.4f}  accuracy={mq['accuracy']:.4f}")
    print(f"  {'-'*64}")

## 11. Results store

In [ ]:
def load_results():
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE) as f:
            return json.load(f)
    return {}

def save_result(results, key, value):
    results[key] = value
    with open(RESULTS_FILE, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  saved {key}")

results = load_results()
print(f"{len(results)} existing entries in {RESULTS_FILE}")

2 existing entries in /content/drive/MyDrive/GNN_MEA/results/matched_noexp_results.json


## 12. Main loop

In [ ]:
def phase1_cached(name, dataset, shadow_idx, model, device):
    path = os.path.join(CACHE_DIR, f'{name}_{ARCH}_phase1.pt')
    if os.path.exists(path):
        blob = torch.load(path, weights_only=False)
        print(f"  Phase 1 cached: {len(blob['pairs'])} pairs, "
              f"{blob['queries']:,} queries")
        return blob['pairs'], blob['non_boundary'], blob['queries']
    ledger = QueryLedger()
    pairs, non_boundary = run_phase1(dataset, shadow_idx, model, ledger, device)
    blob = {'pairs': pairs, 'non_boundary': non_boundary,
            'queries': ledger.api_calls}
    torch.save(blob, path)
    print(f"  Phase 1: {len(pairs)} pairs from {len(shadow_idx)} shadow graphs "
          f"({100*len(pairs)/len(shadow_idx):.1f}%), {ledger.api_calls:,} queries")
    return pairs, non_boundary, ledger.api_calls


def build_hybrid_log(guided_log, dataset, shadow_idx, model, device,
                     phase1_queries, rng):
    order = list(shadow_idx); rng.shuffle(order)
    out, qi = [], 0
    model.eval()
    for k, (o, f, cq) in enumerate(guided_log):
        if qi + 2 > len(order):
            break
        pair = []
        for _ in range(2):
            g = dataset[order[qi]].clone(); qi += 1
            gd = g.to(device)
            with torch.no_grad():
                b = torch.zeros(gd.num_nodes, dtype=torch.long, device=device)
                pred = model(gd.x, gd.edge_index, b).argmax(1).item()
            g = gd.cpu()
            g.y = torch.tensor([pred])
            pair.append(g)
        cum = cq + qi          # one query per random sample labelled so far
        out.append((o, f, cum))
        out.append((pair[0], pair[1], cum, 'random'))
    return out

In [ ]:
for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    for name, cfg in DATASETS.items():
        print(f"\n{'='*70}\n{name} / {ARCH} / GNNExplainer / seed {seed}\n{'='*70}")
        ds = datasets[name]
        model = victims[name]
        shadow_idx = dataset_splits[name]['shadow']
        test_idx = dataset_splits[name]['test']
        in_dim, n_cls = get_feature_dim(ds), ds.num_classes
        balanced_test = get_balanced_test_idx(model, ds, test_idx, device, seed)
        print(f"  balanced test set: {len(balanced_test)} graphs")

        p1_pairs, non_boundary, p1_q = phase1_cached(
            name, ds, shadow_idx, model, device)

        floor = 2 * len(p1_pairs)
        if cfg['target_n_train'] <= floor:
            print(f"  !! WARNING target_n_train={cfg['target_n_train']} <= Phase-1 "
                  f"floor {floor}. Both methods would train on the identical shared "
                  f"Phase-1 set and the comparison is VACUOUS. Raise the target "
                  f"above {floor} or skip this dataset.")
        else:
            differ = (cfg['target_n_train'] - floor) // 2
            print(f"  Phase-1 floor {floor} samples of target "
                  f"{cfg['target_n_train']}: only {differ} of "
                  f"{cfg['target_n_train']//2} pairs differ between methods "
                  f"({200*differ/cfg['target_n_train']:.0f}% of the training set)")

        # ---- guided method ----------------------------------------------
        print("\n  [guided] Phase 2")
        g_log, g_ledger, g_stats = run_arm(
            'guided', name, model, explainers[name], p1_pairs, p1_q,
            non_boundary, ds, cfg, None, seed, device)
        mean_C = float(np.mean(g_stats['cand_sizes'])) if g_stats['cand_sizes'] else 40.0
        print(f"  [guided] pairs={len(g_log)} queries={g_ledger.api_calls:,} "
              f"mean|C|={mean_C:.1f} explanations={g_ledger.explanations_consumed:,}")

        # ---- matched random method --------------------------------------
        fallback = int(round(mean_C))
        cand_plan = defaultdict(lambda: [fallback])
        cand_plan.update({i: s for i, s in g_stats['cand_by_graph'].items() if s})
        print(f"\n  [no-exp] Phase 2, |C| matched per graph "
              f"({len(cand_plan)} graphs planned, fallback {fallback})")
        n_log, n_ledger, n_stats = run_arm(
            'random', name, model, None, p1_pairs, p1_q,
            non_boundary, ds, cfg, cand_plan, seed, device)
        n_meanC = (float(np.mean(n_stats['cand_sizes']))
                   if n_stats['cand_sizes'] else 0.0)
        print(f"  [no-exp] pairs={len(n_log)} queries={n_ledger.api_calls:,} "
              f"mean|C|={n_meanC:.1f} (guided {mean_C:.1f})")

        # ---- hybrid (derived) -------------------------------------------
        h_log = build_hybrid_log(g_log, ds, shadow_idx, model, device, p1_q,
                                 np.random.default_rng(seed))

        arms = {'Boundary': (g_log, g_ledger, g_stats),
                'No-Exp':   (n_log, n_ledger, n_stats),
                'Hybrid':   (h_log, g_ledger, g_stats)}

        # ---- Q_cap ------------------------------------------------------
        target = cfg['target_n_train']
        at_n = {}
        for arm, (log, led, st) in arms.items():
            samples, q_needed, reached = truncate_at_n(
                [(a, b, c) for (a, b, c, *_rest) in log], target)
            at_n[arm] = dict(samples=samples, queries=q_needed, reached=reached)
            print(f"  {arm:9s} -> reaches n_train={len(samples)} at {q_needed:,} "
                  f"queries  reached={reached}")

        reached_q = [v['queries'] for v in at_n.values() if v['reached']]
        q_cap = max(reached_q) if reached_q else max(
            led.api_calls for (_l, led, _s) in arms.values())
        print(f"\n  Q_cap = {q_cap:,}")

        # ---- train + evaluate at matched Q ------------------------------
        for arm, (log, led, st) in arms.items():
            flat = [(a, b, c) for (a, b, c, *_rest) in log]
            n_random_pairs = sum(1 for e in log if len(e) > 3 and e[3] == 'random')
            row = {
                'dataset': name, 'arch': ARCH, 'explainer': EXPLAINER_NAME,
                'arm': arm, 'seed': seed,
                'search_stats_source': 'Boundary' if arm == 'Hybrid' else arm,
                'target_n_train': target,
                'q_cap': q_cap,
                'queries_phase1': p1_q,
                'queries_total': led.api_calls,
                'explanations_consumed': led.explanations_consumed,
                'explanation_rate': round(led.explanations_consumed /
                                          max(led.api_calls, 1), 6),
                'n_pairs_phase1': len(p1_pairs),
                'n_pairs_total': len(flat),
                'n_random_shadow_pairs': n_random_pairs,
                'n_pairs_phase2': len(flat) - len(p1_pairs) - n_random_pairs,
                'graphs_attempted_phase2': sum(st['exits'].values()),
                'exit_counts': dict(st['exits']),
                'mean_iters_to_success': (float(np.mean(st['iters_success']))
                                          if st['iters_success'] else None),
                'mean_candidate_set_size': (float(np.mean(st['cand_sizes']))
                                            if st['cand_sizes'] else None),
                'phase2_success_rate': (round(st['exits'].get('success', 0) /
                                              max(sum(st['exits'].values()), 1), 4)),
                'mean_full_space': (float(np.mean(st['full_space']))
                                    if st['full_space'] else None),
                'candidate_reduction_factor': (
                    round(float(np.mean(st['full_space'])) /
                          float(np.mean(st['cand_sizes'])), 2)
                    if st['cand_sizes'] and st['full_space'] else None),
                'mean_vsub_frac': (float(np.mean(st['vsub_frac']))
                                   if st['vsub_frac'] else None),
                'stopped_reason': st['stopped_reason'],
                'wall_seconds': st['wall_s'],
                'ceiling_hit': led.ceiling_hit,
            }

            s_n = at_n[arm]
            row['queries_to_target'] = s_n['queries']
            row['reached_target'] = s_n['reached']

            # Reading 1 - matched training-set size (equal data).
            if s_n['samples']:
                surr = train_surrogate(s_n['samples'], in_dim, n_cls, ARCH,
                                       device, seed=seed)
                f, a = evaluate(surr, model, ds, balanced_test, device)
                row['matched_n_train'] = {'n_train': len(s_n['samples']),
                                          'fidelity': f, 'accuracy': a}

            # Reading 2 - matched query budget (equal victim queries).
            s_q = truncate_at_q(flat, q_cap)
            if s_q:
                surr = train_surrogate(s_q, in_dim, n_cls, ARCH, device, seed=seed)
                f, a = evaluate(surr, model, ds, balanced_test, device)
                row['matched_queries'] = {'n_train': len(s_q),
                                          'fidelity': f, 'accuracy': a}

            print_arm_summary(row)
            save_result(results, f"{name}_{ARCH}_{arm}_{EXPLAINER_NAME}_seed{seed}", row)

print("\nAll runs complete.")